# IN HIS NAME
## NNDL - HW Extra - Q3

# Part 3-3-1

In [2]:
from datasets import load_dataset, Dataset
from collections import Counter
import pandas as pd
import numpy as np

DS_NAME = 'emotion'
DS_TRAINING_SIZE = 1500
DS_TEST_SIZE = 100
DS_VALIDATION_SIZE = 50

dataset = load_dataset(DS_NAME)

def stratified_sample(dataset_split, n_samples, seed=42):
    df = pd.DataFrame(dataset_split)
    
    label_counts = df['label'].value_counts(normalize=True)
    
    sampled_dfs = []
    for label, prop in label_counts.items():
        n_label = int(round(prop * n_samples))
        df_label = df[df['label'] == label]
        sampled_df_label = df_label.sample(n=n_label, random_state=seed)
        sampled_dfs.append(sampled_df_label)
    
    sampled_df = pd.concat(sampled_dfs).sample(frac=1, random_state=seed).reset_index(drop=True)
    return Dataset.from_pandas(sampled_df)

train_sampled = stratified_sample(dataset['train'], DS_TRAINING_SIZE)
test_sampled = stratified_sample(dataset['test'], DS_TEST_SIZE)
valid_sampled = stratified_sample(dataset['validation'], DS_VALIDATION_SIZE)

def show_label_distribution(dataset_split, name):
    counter = Counter(dataset_split['label'])
    total = sum(counter.values())
    print(f"Label distribution in {name}:")
    for label, count in counter.items():
        print(f"  Label {label}: {count} ({count/total:.2%})")
    print()

show_label_distribution(train_sampled, "Train Sampled")
show_label_distribution(test_sampled, "Test Sampled")
show_label_distribution(valid_sampled, "Validation Sampled")

show_label_distribution(dataset['train'], "Train Original")
show_label_distribution(dataset['test'], "Test Original")
show_label_distribution(dataset['validation'], "Validation Original")


Label distribution in Train Sampled:
  Label 3: 202 (13.47%)
  Label 2: 122 (8.13%)
  Label 1: 503 (33.53%)
  Label 0: 437 (29.13%)
  Label 4: 182 (12.13%)
  Label 5: 54 (3.60%)

Label distribution in Test Sampled:
  Label 4: 11 (11.00%)
  Label 0: 29 (29.00%)
  Label 3: 14 (14.00%)
  Label 1: 35 (35.00%)
  Label 2: 8 (8.00%)
  Label 5: 3 (3.00%)

Label distribution in Validation Sampled:
  Label 1: 18 (36.00%)
  Label 4: 5 (10.00%)
  Label 0: 14 (28.00%)
  Label 2: 4 (8.00%)
  Label 5: 2 (4.00%)
  Label 3: 7 (14.00%)

Label distribution in Train Original:
  Label 0: 4666 (29.16%)
  Label 3: 2159 (13.49%)
  Label 2: 1304 (8.15%)
  Label 5: 572 (3.57%)
  Label 4: 1937 (12.11%)
  Label 1: 5362 (33.51%)

Label distribution in Test Original:
  Label 0: 581 (29.05%)
  Label 1: 695 (34.75%)
  Label 4: 224 (11.20%)
  Label 3: 275 (13.75%)
  Label 2: 159 (7.95%)
  Label 5: 66 (3.30%)

Label distribution in Validation Original:
  Label 0: 550 (27.50%)
  Label 2: 178 (8.90%)
  Label 3: 275 (13.7

# Part 3-3-2

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "meta-llama/Llama-3.2-1B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",  
    low_cpu_mem_usage=True  
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {MODEL_NAME}")
print(f"Total parameters: {total_params/1e6:.2f}M")


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B.
401 Client Error. (Request ID: Root=1-69953abb-65de6a116aca9194437077da;591c71e4-d87f-43b2-83ad-a0a14ab25990)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-1B is restricted. You must have access to it and be authenticated to access it. Please log in.

# Part 3-3-3

In [5]:
def format_prompt(system_instruction: str, user_input: str, assistant_output: str) -> str:

    prompt = (
        f"<s>[INST] {system_instruction} [/INST] {user_input} </s>\n"
        f"[ASSISTANT] {assistant_output} </s>"
    )
    return prompt

system_instruction = "شما یک مدل تشخیص احساسات هستید. برچسب‌های ممکن: sadness, joy, love, anger, fear, surprise."
user_input = "I feel amazing today!"
assistant_output = "joy"

formatted_prompt = format_prompt(system_instruction, user_input, assistant_output)
print(formatted_prompt)


<s>[INST] شما یک مدل تشخیص احساسات هستید. برچسب‌های ممکن: sadness, joy, love, anger, fear, surprise. [/INST] I feel amazing today! </s>
[ASSISTANT] joy </s>


# Part 3-3-4

In [6]:
sample_data = {
    "system_instruction": "شما یک مدل تشخیص احساسات هستید. برچسب‌های ممکن: sadness, joy, love, anger, fear, surprise.",
    "user_input": "I feel amazing today!",
    "assistant_output": "joy"
}

def format_prompt(system_instruction, user_input, assistant_output):
    return (
        f"<s>[INST] {system_instruction} [/INST] {user_input} </s>\n"
        f"[ASSISTANT] {assistant_output} </s>"
    )

prompt = format_prompt(
    sample_data["system_instruction"], 
    sample_data["user_input"], 
    sample_data["assistant_output"]
)

print("Formatted Prompt:\n", prompt)

tokenized = tokenizer(
    prompt,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"  
)

print("\nToken IDs:\n", tokenized["input_ids"])

decoded_text = tokenizer.decode(tokenized["input_ids"][0], skip_special_tokens=True)
print("\nDecoded Text:\n", decoded_text)


Formatted Prompt:
 <s>[INST] شما یک مدل تشخیص احساسات هستید. برچسب‌های ممکن: sadness, joy, love, anger, fear, surprise. [/INST] I feel amazing today! </s>
[ASSISTANT] joy </s>


NameError: name 'tokenizer' is not defined